# Sparse3D quickstart

Reconstruct an object from a handful of photos with the Python API, inspect the camera poses and the point cloud,
and read the metrics. Run it from the `notebooks/` folder of a checkout with the requirements installed and COLMAP
available (see the README).

In [ ]:
import sys, tempfile
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.reconstruction.pipeline import ReconstructionPipeline

images = sorted(Path('../data/samples/dtu_scan1_6views').glob('*.png'))   # or any 3-6 photos
work = Path(tempfile.mkdtemp(prefix='sparse3d_'))

pipe = ReconstructionPipeline(
    job_id='quickstart', input_image_paths=images,
    workspace_dir=work / 'ws', output_dir=work / 'out',
    options={'recon_method': 'classical', 'num_views': 6, 'quality': 'standard'},
    on_log=print)
report = pipe.run()
print(report['status'], report['error_message'])

## Metrics

Values that need a reference model are `None`. Pass `ground_truth_path=` (and optionally `gt_projections=`) to the
pipeline to measure them.

In [ ]:
result = report['results']['classical']
result['metrics']

## Point cloud and cameras

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import open3d as o3d

pcd = o3d.io.read_point_cloud(str(work / 'out' / 'quickstart' / 'point_cloud.ply'))
pts, cols = np.asarray(pcd.points), np.asarray(pcd.colors)
cams = np.array([c['position'] for c in result['cameras']])

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection='3d')
ax.scatter(*pts.T, c=cols if len(cols) else 'gray', s=4)
ax.scatter(*cams.T, c='red', marker='^', s=60, label='cameras')
ax.legend()
plt.show()